In [1]:
from pathlib import Path

import gc
import optuna
import pandas as pd
from catboost import CatBoostRegressor

from train_model import train_model_and_validation_catboost


In [2]:
DATA_DIR = Path("../data_classic")

validation_file = (
    DATA_DIR
    / "train_3_predict_from_2025-12-15.parquet"
)

competition_test_file = (
    DATA_DIR
    / "competition_test.parquet"
)

train_files = [
    DATA_DIR / "train_1_predict_from_2025-10-16.parquet",
    DATA_DIR / "train_2_predict_from_2025-11-15.parquet",
]

final_train_files = [
    DATA_DIR / "train_1_predict_from_2025-10-16.parquet",
    DATA_DIR / "train_2_predict_from_2025-11-15.parquet",
    DATA_DIR / "train_3_predict_from_2025-12-15.parquet",
    DATA_DIR / "test_for_us_predict_from_2026-01-14.parquet",
]


In [3]:
importance = (
    pd.read_csv("../hren.csv")
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

selected_features = importance["features"].head(500).tolist()

gc.collect()

print(len(selected_features))


500


In [4]:
def objective(trial):
    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
        "depth": trial.suggest_int("depth", 6, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 12.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
        "random_seed": 42,
        "thread_count": -1,
        "allow_writing_files": False,
    }

    model = CatBoostRegressor(**params)

    valid_prediction, rmsle = train_model_and_validation_catboost(
        model=model,
        train_files=train_files,
        validation_file=validation_file,
        validation=True,
        selected_features=selected_features,
        use_user_id=False,
        verbose=500,
    )

    trial.set_user_attr(
        "best_iteration",
        model.best_iteration_ + 1,
    )

    del model
    del valid_prediction
    gc.collect()

    return rmsle


In [5]:
N_TRIALS = 8

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
)

print(study.best_value)
print(study.best_params)
print(study.best_trial.user_attrs)


[I 2026-08-18 22:54:34,409] A new study created in memory with name: no-name-eabfac20-d3b0-482c-9d8e-2bfc7e51863f


Train: (500000, 501)
Validation/Test: (250000, 502)
0:	learn: 2.3156700	test: 2.3276651	best: 2.3276651 (0)	total: 880ms	remaining: 29m 19s
500:	learn: 1.6961153	test: 1.7417445	best: 1.7417428 (498)	total: 5m 26s	remaining: 16m 15s
1000:	learn: 1.6762618	test: 1.7412396	best: 1.7412349 (997)	total: 8m 59s	remaining: 8m 58s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 1.741223337
bestIteration = 1023

Shrink model to first 1024 iterations.


[I 2026-08-18 23:05:27,475] Trial 0 finished with value: 1.7412216091886294 and parameters: {'learning_rate': 0.031854305348131315, 'depth': 9, 'l2_leaf_reg': 9.31993941811405, 'random_strength': 1.1973169683940732, 'bagging_temperature': 0.31203728088487304, 'rsm': 0.662397808134481}. Best is trial 0 with value: 1.7412216091886294.


Train: (500000, 501)
Validation/Test: (250000, 502)
0:	learn: 2.3302958	test: 2.3422395	best: 2.3422395 (0)	total: 574ms	remaining: 19m 8s
500:	learn: 1.7057762	test: 1.7425327	best: 1.7425327 (500)	total: 3m 38s	remaining: 10m 54s
1000:	learn: 1.6930950	test: 1.7413183	best: 1.7413183 (1000)	total: 7m 1s	remaining: 7m
1500:	learn: 1.6814666	test: 1.7411132	best: 1.7411124 (1499)	total: 10m 15s	remaining: 3m 24s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 1.741094704
bestIteration = 1560

Shrink model to first 1561 iterations.


[I 2026-08-18 23:17:13,159] Trial 1 finished with value: 1.741093527079032 and parameters: {'learning_rate': 0.017613762547568974, 'depth': 9, 'l2_leaf_reg': 8.011150117432088, 'random_strength': 1.416145155592091, 'bagging_temperature': 0.041168988591604894, 'rsm': 0.9879639408647978}. Best is trial 1 with value: 1.741093527079032.


...

Train: (500000, 501)
Validation/Test: (250000, 502)
0:	learn: 2.3195198	test: 2.3315675	best: 2.3315675 (0)	total: 221ms	remaining: 7m 22s
500:	learn: 1.7104038	test: 1.7423901	best: 1.7423900 (499)	total: 1m 45s	remaining: 5m 15s
1000:	learn: 1.7028238	test: 1.7413724	best: 1.7413717 (999)	total: 3m 25s	remaining: 3m 24s
1500:	learn: 1.6968009	test: 1.7411451	best: 1.7411376 (1496)	total: 5m 1s	remaining: 1m 40s
1999:	learn: 1.6909124	test: 1.7410717	best: 1.7410502 (1896)	total: 6m 38s	remaining: 0us

bestTest = 1.74105017
bestIteration = 1896

Shrink model to first 1897 iterations.


[I 2026-08-18 23:43:12,576] Trial 6 finished with value: 1.7410461615069073 and parameters: {'learning_rate': 0.02870761961280168, 'depth': 6, 'l2_leaf_reg': 8.842330265121568, 'random_strength': 0.8803049874792026, 'bagging_temperature': 0.24407646968955765, 'rsm': 0.798070764044508}. Best is trial 6 with value: 1.7410461615069073.


Train: (500000, 501)
Validation/Test: (250000, 502)
0:	learn: 2.3313743	test: 2.3433559	best: 2.3433559 (0)	total: 479ms	remaining: 15m 57s
500:	learn: 1.7058295	test: 1.7426451	best: 1.7426451 (500)	total: 4m 1s	remaining: 12m 2s
1000:	learn: 1.6928406	test: 1.7414752	best: 1.7414752 (1000)	total: 7m 49s	remaining: 7m 48s
1500:	learn: 1.6805816	test: 1.7411355	best: 1.7411186 (1485)	total: 11m 28s	remaining: 3m 48s
1999:	learn: 1.6691379	test: 1.7410671	best: 1.7410324 (1856)	total: 15m 10s	remaining: 0us

bestTest = 1.741032425
bestIteration = 1856

Shrink model to first 1857 iterations.


[I 2026-08-18 23:58:31,040] Trial 7 finished with value: 1.741030270190103 and parameters: {'learning_rate': 0.01654748345018483, 'depth': 9, 'l2_leaf_reg': 4.587799816000169, 'random_strength': 1.325044568707964, 'bagging_temperature': 0.6234221521788219, 'rsm': 0.8080272084711243}. Best is trial 7 with value: 1.741030270190103.


1.741030270190103
{'learning_rate': 0.01654748345018483, 'depth': 9, 'l2_leaf_reg': 4.587799816000169, 'random_strength': 1.325044568707964, 'bagging_temperature': 0.6234221521788219, 'rsm': 0.8080272084711243}
{'best_iteration': 1857}


In [6]:
best_params = study.best_params.copy()
best_iterations = study.best_trial.user_attrs["best_iteration"]

In [7]:
final_catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=best_iterations,
    random_seed=42,
    thread_count=-1,
    allow_writing_files=False,
    **best_params,
)

catboost_submission = train_model_and_validation_catboost(
    model=final_catboost_model,
    train_files=final_train_files,
    validation_file=competition_test_file,
    validation=False,
    selected_features=selected_features,
    use_user_id=False,
    verbose=100,
)

catboost_submission.to_csv(
    "submission_catboost_optuna.csv",
    index=False,
)

catboost_submission.head()


Train: (1000000, 501)
Validation/Test: (250000, 501)
0:	learn: 2.3201480	total: 1.73s	remaining: 53m 32s
100:	learn: 1.7514208	total: 1m 30s	remaining: 26m 4s
200:	learn: 1.7178687	total: 2m 48s	remaining: 23m 7s
300:	learn: 1.7128381	total: 4m 4s	remaining: 21m 1s
400:	learn: 1.7104042	total: 5m 17s	remaining: 19m 13s
500:	learn: 1.7085120	total: 6m 31s	remaining: 17m 39s
600:	learn: 1.7068711	total: 7m 45s	remaining: 16m 12s
700:	learn: 1.7053132	total: 8m 58s	remaining: 14m 48s
800:	learn: 1.7038380	total: 10m 26s	remaining: 13m 46s
900:	learn: 1.7021077	total: 11m 39s	remaining: 12m 22s
1000:	learn: 1.7003337	total: 13m 1s	remaining: 11m 8s
1100:	learn: 1.6985996	total: 15m 1s	remaining: 10m 19s
1200:	learn: 1.6969458	total: 16m 42s	remaining: 9m 7s
1300:	learn: 1.6953896	total: 18m 3s	remaining: 7m 42s
1400:	learn: 1.6938483	total: 19m 28s	remaining: 6m 20s
1500:	learn: 1.6923999	total: 21m 9s	remaining: 5m 1s
1600:	learn: 1.6910135	total: 22m 14s	remaining: 3m 33s
1700:	learn: 1.

,user_id,predict
0,2,2.079074
1,7,87.434307
2,15,7.408961
3,18,142.867978
4,23,0.413992
